The baseline experiments showed that description + tags provides the strongest text representation, reaching approximately 90% accuracy under stratified cross-validation.

This notebook investigates whether performance can be improved further through:

- tuning the LinearSVC classifier;
- character-level TF-IDF features;
- combinations of word- and character-level features;
- comparison with logistic regression.

All experiments use the same cleaned dataset and 5-fold stratified cross-validation.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedKFold

from chef_classifier.data import (
    clean_training_data,
    load_training_data,
)
from chef_classifier.features import combine_text_fields

In [ ]:
PROJECT_ROOT = Path("..")
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"

train = load_training_data(TRAIN_PATH)
train_clean = clean_training_data(train)

X = combine_text_fields(
    train_clean,
    ["description", "tags"],
)

y = train_clean["chef_id"]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

## LinearSVC Regularization Parameter Tuning

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

In [ ]:
pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            LinearSVC(),
        ),
    ]
)

param_grid = {
    "classifier__C": [
        0.01,
        0.1,
        0.5,
        1.0,
        2.0,
        5.0,
        10.0,
    ]
}

In [ ]:
grid_search_c = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
)

grid_search_c.fit(X, y)

In [ ]:
grid_search_c.best_params_

In [ ]:
grid_search_c.best_score_

In [ ]:
c_results = pd.DataFrame(
    grid_search_c.cv_results_
)[
    [
        "param_classifier__C",
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
].sort_values("param_classifier__C")

c_results

## Character-level IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

In [ ]:
char_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                analyzer="char",
                ngram_range=(3, 5),
                min_df=2,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            LinearSVC(C=5.0),
        ),
    ]
)

char_scores = cross_val_score(
    char_pipeline,
    X,
    y,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

char_scores

In [ ]:
char_scores.mean(), char_scores.std()

In [ ]:
comparison = pd.DataFrame(
    {
        "model": [
            "Word TF-IDF",
            "Character TF-IDF",
        ],
        "cv_mean_accuracy": [
            grid_search_c.best_score_,
            char_scores.mean(),
        ],
        "cv_std_accuracy": [
            grid_search_c.cv_results_["std_test_score"][
                grid_search_c.best_index_
            ],
            char_scores.std(),
        ],
    }
)

comparison

## Combined word- and character-level TF-IDF


In [ ]:
from sklearn.pipeline import FeatureUnion, Pipeline

In [ ]:
combined_text_pipeline = Pipeline(
    [
        (
            "features",
            FeatureUnion(
                [
                    (
                        "word",
                        TfidfVectorizer(
                            analyzer="word",
                            ngram_range=(1, 2),
                            min_df=2,
                            sublinear_tf=True,
                        ),
                    ),
                    (
                        "char",
                        TfidfVectorizer(
                            analyzer="char",
                            ngram_range=(3, 5),
                            min_df=2,
                            sublinear_tf=True,
                        ),
                    ),
                ]
            ),
        ),
        (
            "classifier",
            LinearSVC(C=5.0),
        ),
    ]
)

combined_scores = cross_val_score(
    combined_text_pipeline,
    X,
    y,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

combined_scores

combined_scores.mean(), combined_scores.std()

In [ ]:
comparison = pd.DataFrame(
    {
        "model": [
            "Word TF-IDF",
            "Character TF-IDF",
            "Word + Character TF-IDF",
        ],
        "cv_mean_accuracy": [
            grid_search_c.best_score_,
            char_scores.mean(),
            combined_scores.mean(),
        ],
        "cv_std_accuracy": [
            grid_search_c.cv_results_["std_test_score"][
                grid_search_c.best_index_
            ],
            char_scores.std(),
            combined_scores.std(),
        ],
    }
).sort_values(
    "cv_mean_accuracy",
    ascending=False,
)

comparison

## Logistic Regression comparison

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg_pipeline = Pipeline(
    [
        (
            "features",
            FeatureUnion(
                [
                    (
                        "word",
                        TfidfVectorizer(
                            analyzer="word",
                            ngram_range=(1, 2),
                            min_df=2,
                            sublinear_tf=True,
                        ),
                    ),
                    (
                        "char",
                        TfidfVectorizer(
                            analyzer="char",
                            ngram_range=(3, 5),
                            min_df=2,
                            sublinear_tf=True,
                        ),
                    ),
                ]
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
            ),
        ),
    ]
)

logreg_scores = cross_val_score(
    logreg_pipeline,
    X,
    y,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

logreg_scores

In [ ]:
logreg_scores.mean(), logreg_scores.std()

In [ ]:
comparison = pd.DataFrame(
    {
        "model": [
            "Word TF-IDF + LinearSVC",
            "Character TF-IDF + LinearSVC",
            "Word + Character TF-IDF + LinearSVC",
            "Word + Character TF-IDF + Logistic Regression",
        ],
        "cv_mean_accuracy": [
            grid_search_c.best_score_,
            char_scores.mean(),
            combined_scores.mean(),
            logreg_scores.mean(),
        ],
        "cv_std_accuracy": [
            grid_search_c.cv_results_["std_test_score"][
                grid_search_c.best_index_
            ],
            char_scores.std(),
            combined_scores.std(),
            logreg_scores.std(),
        ],
    }
).sort_values(
    "cv_mean_accuracy",
    ascending=False,
)

comparison

In [ ]:
logreg_param_grid = {
    "classifier__C": [
        0.01,
        0.1,
        0.5,
        1.0,
        2.0,
        5.0,
        10.0,
        20.0,
    ]
}

logreg_grid_search = GridSearchCV(
    logreg_pipeline,
    param_grid=logreg_param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
)

logreg_grid_search.fit(X, y)

In [ ]:
logreg_grid_search.best_params_

In [ ]:
logreg_grid_search.best_score_

In [ ]:
logreg_c_results = pd.DataFrame(
    logreg_grid_search.cv_results_
)[
    [
        "param_classifier__C",
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
].sort_values("param_classifier__C")

logreg_c_results

## Final model error analysis

The combined word and character-level TF-IDF representation with LinearSVC achieved the strongest cross-validation performance.

To understand the remaining errors without relying on a single validation split, out-of-fold predictions are generated using the same 5-fold stratified cross-validation procedure.

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.model_selection import cross_val_predict

from chef_classifier.evaluation import calculate_accuracy

oof_predictions = cross_val_predict(
    combined_text_pipeline,
    X,
    y,
    cv=cv,
    n_jobs=-1,
)

calculate_accuracy(y, oof_predictions)

In [ ]:
print(classification_report(y, oof_predictions))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y,
    oof_predictions,
)

In [ ]:
oof_results = train_clean.copy()

oof_results["predicted_chef"] = oof_predictions
oof_results["correct"] = (
    oof_results["chef_id"] == oof_results["predicted_chef"]
)

errors = oof_results[~oof_results["correct"]].copy()

In [ ]:
confusion_pairs = (
    errors.groupby(["chef_id", "predicted_chef"])
    .size()
    .sort_values(ascending=False)
)

confusion_pairs.head(15)

Overall, the remaining errors are concentrated among a few chef pairs rather than being uniformly distributed. This suggests that these chefs share more similar recipe categories or linguistic patterns than the easier-to-identify classes.